In [ ]:
# import re

# def load_urls(file_path):
#     with open(file_path, "r", encoding="utf-8") as f:
#         return [line.strip() for line in f if line.strip()]

# def normalize_url(url):
#     url = url.strip()

#     # thêm http nếu thiếu
#     if not url.startswith("http://") and not url.startswith("https://"):
#         url = "http://" + url

#     return url

# def is_valid_url(url):
#     pattern = re.compile(
#         r'^(http|https)://'          # protocol
#         r'([a-zA-Z0-9-]+\.)+'       # domain
#         r'[a-zA-Z]{2,}'             # TLD
#     )
#     return re.match(pattern, url) is not None

# def clean_urls(urls):
#     cleaned = set()

#     for url in urls:
#         url = normalize_url(url)

#         if not is_valid_url(url):
#             print(f"[INVALID] {url}")
#             continue

#         cleaned.add(url)

#     return list(cleaned)

# def save_urls(urls, output_file):
#     with open(output_file, "w", encoding="utf-8") as f:
#         for url in urls:
#             f.write(url + "\n")

# def main():
#     raw_urls = load_urls("urls.txt")
#     print(f"Raw URLs: {len(raw_urls)}")

#     clean = clean_urls(raw_urls)
#     print(f"Clean URLs: {len(clean)}")

#     save_urls(clean, "urls_clean.txt")

# if __name__ == "__main__":
#     main()

Check alive 

In [ ]:
# import requests
# import time

# HEADERS = {
#     "User-Agent": "Mozilla/5.0"
# }

# def load_urls(file_path):
#     with open(file_path, "r", encoding="utf-8") as f:
#         return [line.strip() for line in f if line.strip()]

# def is_alive(url):
#     try:
#         res = requests.head(url, headers=HEADERS, timeout=3, allow_redirects=True)
#         return res.status_code < 400
#     except:
#         return False

# def check_alive_urls(urls):
#     alive = []

#     for url in urls:
#         if is_alive(url):
#             print(f"[ALIVE] {url}")
#             alive.append(url)
#         else:
#             print(f"[DEAD] {url}")

#         time.sleep(0.5)

#     return alive

# def save_urls(urls, file_path):
#     with open(file_path, "w", encoding="utf-8") as f:
#         for url in urls:
#             f.write(url + "\n")

# def main():
#     urls = load_urls("urls_clean.txt")
#     print(f"Total clean URLs: {len(urls)}")

#     alive_urls = check_alive_urls(urls)
#     print(f"Alive URLs: {len(alive_urls)}")

#     save_urls(alive_urls, "urls_alive.txt")

# if __name__ == "__main__":
#     main()

In [ ]:
import re
import requests
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def normalize_url(url):
    url = url.strip()
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "http://" + url
    return url

def is_valid_url(url):
    pattern = re.compile(
        r'^(http|https)://'
        r'([a-zA-Z0-9-]+\.)+'
        r'[a-zA-Z]{2,}'
    )
    return re.match(pattern, url) is not None

def is_alive(url):
    try:
        res = requests.head(url, headers=HEADERS, timeout=3, allow_redirects=True)
        return res.status_code < 400
    except:
        return False

def process_urls(urls):
    alive = []
    not_found = []

    seen = set()

    for url in urls:
        url = normalize_url(url)

        if url in seen:
            continue
        seen.add(url)

        if not is_valid_url(url):
            print(f"[INVALID] {url}")
            not_found.append(url)
            continue

        if is_alive(url):
            print(f"[ALIVE] {url}")
            alive.append(url)
        else:
            print(f"[DEAD] {url}")
            not_found.append(url)

        time.sleep(0.5)

    return alive, not_found

def save_urls(urls, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")

def main():
    urls = load_urls("urls.txt")
    print(f"Total raw URLs: {len(urls)}")

    alive, not_found = process_urls(urls)

    print(f"Alive: {len(alive)}")
    print(f"Not found (invalid + dead): {len(not_found)}")

    save_urls(alive, "urls_alive.txt")
    save_urls(not_found, "not_found.txt")

if __name__ == "__main__":
    main()

In [ ]:
import re
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

MAX_WORKERS = 20
RETRY = 2
TIMEOUT = 3

def load_urls(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def normalize_url(url):
    url = url.strip()
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "http://" + url
    return url.lower()

def is_valid_url(url):
    try:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and parsed.netloc != ""
    except:
        return False

def check_alive(url):
    for _ in range(RETRY):
        try:
            res = requests.head(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True)
            if res.status_code < 400:
                return "ALIVE"
            if res.status_code in (403, 405):
                res = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
                if res.status_code < 400:
                    return "ALIVE"
            return "DEAD"
        except:
            continue
    return "DEAD"

def process_single(url):
    url = normalize_url(url)

    if not is_valid_url(url):
        return ("INVALID", url)

    status = check_alive(url)
    return (status, url)

def process_urls(urls):
    alive = []
    not_found = []
    seen = set()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []

        for url in urls:
            url = normalize_url(url)
            if url in seen:
                continue
            seen.add(url)
            futures.append(executor.submit(process_single, url))

        for future in as_completed(futures):
            status, url = future.result()

            if status == "ALIVE":
                print(f"[ALIVE] {url}")
                alive.append(url)
            else:
                print(f"[{status}] {url}")
                not_found.append(url)

    return alive, not_found

def save_urls(urls, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for url in urls:
            f.write(url + "\n")

def main():
    urls = load_urls("urls.txt")
    print(f"Total raw URLs: {len(urls)}")

    alive, not_found = process_urls(urls)

    print(f"Alive: {len(alive)}")
    print(f"Not found (invalid + dead): {len(not_found)}")

    save_urls(alive, "urls_alive.txt")
    save_urls(not_found, "not_found.txt")

if __name__ == "__main__":
    main()